# SI-noise study: giving ε real noise to scale

Follow-up to notebook 22 (`slurm/si_schedules/`). There, ε barely changed the linear, trig and VP interpolants: with
γ = 0 the path's only noise is the base's own, Γ_t = α_t·σ ≤ σ = 0.10, and the SDE's noise is √(2ε)·Γ_t, so even
ε = 1 injected almost nothing (calibration moved by < 0.03). Only the bridge (γ > 0) responded, and its extra spread
came with 3-5× worse melt-pool error. This study (`slurm/si_noise/`) asks whether **real** path noise buys diversity
**without** losing accuracy.

| phase | what | trained? | evals |
|---|---|---|---|
| 0 | the SI-schedules `linear·enc` model, Euler / Heun at ε = 2, 5, 10, 20 (ε_t = ε·Γ_t² as before; stable at 20 steps) | no | 8 |
| 1 | linear path + **latent noise** γ_t = c·t(1−t), c = 0.5 / 1 / 2 (peak 0.125 / 0.25 / 0.5), σ = 0.10 | 3 runs | 3 × 13 |
| 1 | linear path, γ = 0, **larger base noise** σ = 0.25 / 0.5 | 2 runs | 2 × 13 |

Phase 1 evals: Euler and Heun at ε = 0, 0.1, 0.5, 1, 2, 5, and RK4 at ε = 0. Every run uses the encoder start (better than
bicubic in every SI-schedules arm) and the frozen encoder's features at every network evaluation, as DiffusionSR does.
γ_t = c·t(1−t) vanishes linearly at t = 1, so unlike the bridge its Γ̇ stays bounded (no singularity for Euler/Heun).

**Same scoring as notebooks 21 and 22** (test split, K = 10, 20 steps, `compute_metrics`, overlay scales), **plus** three
ensemble-verification scores that calibration alone cannot replace, because calibration rewards any added spread, even
spread that is just error:

- **CRPS** (fair, K-corrected): one error score for the whole ensemble; lowest only when the ensemble's spread matches
  how wrong it really is. With K = 1 it equals the MAE.
- **Spread-skill ratio (SSR)**: ensemble spread / RMSE of the ensemble mean (K-corrected). ≈ 1 honest, < 1
  under-dispersed (overconfident), > 1 over-dispersed.
- **Rank histogram**: where the truth falls among the 10 sorted samples. Flat = consistent, U = under-dispersed,
  hump = over-dispersed.

Each is reported over all pixels and over the true melt pool (> 1900 K). For the baselines and the SI-schedules evals
they were back-filled from the saved predictions (`analysis/add_skill_metrics.py`, `skill.json`; the original
`results.json` files are untouched).

**Success** = calibration ≳ 0.80 and SSR ≈ 1 with melt-pool depth MAE no worse than DiffusionSR's (14.2 µm).

In [ ]:
# -- Configuration ----------------------------------------------------------------
import os, sys
from pathlib import Path

REPO       = Path(os.getenv('SI_REPO', '/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom'))
RUNS_ROOT  = REPO / 'diffusionsr' / 'runs' / 'si_noise'
EVAL_ROOT  = REPO / 'eval_results' / 'si_noise'
SCHED_RUNS = REPO / 'diffusionsr' / 'runs' / 'si_schedules'     # SI-schedules study (notebook 22)
SCHED_EVAL = REPO / 'eval_results' / 'si_schedules'
PREV_RUNS  = REPO / 'diffusionsr' / 'runs' / 'interpolant'      # previous SI run (notebook 21)
PREV_EVAL  = REPO / 'eval_results' / 'interpolant'
CACHE      = REPO / 'eval_results' / '_predictions_cache'      # SHARED with notebooks 21 and 22
WANDB_ENTITY  = os.getenv('WANDB_ENTITY') or 'ngng-'
WANDB_PROJECT = 'Flow3D_SuperResolution'

# Identical to notebooks 21 and 22, so figures compare one to one.
VMIN_T, VMAX_T = 293, 5000
MP_THRESHOLD_K = 1900.0
CH = 0

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

In [ ]:
# -- Imports -----------------------------------------------------------------------
import json, tempfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
try:
    import ipywidgets as widgets
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
# NB24_STATIC=1 renders fixed frames instead of sliders, for a pre-executed copy.
if os.getenv('NB24_STATIC'):
    HAVE_WIDGETS = False
warnings.filterwarnings('ignore', category=RuntimeWarning)

In [ ]:
# -- Arms --------------------------------------------------------------------------
def eps_tag(e):                       # must match evaluate_si_schedules.eps_tag
    return 'eps' + f'{e:g}'.replace('.', 'p')

S1_EPS = [0.0, 0.01, 0.05, 0.1, 0.25, 0.5, 1.0]        # SI-schedules sweep
P0_EPS = [2.0, 5.0, 10.0, 20.0]                         # Phase 0 extension of the same model
P1_EPS = [0.0, 0.1, 0.5, 1.0, 2.0, 5.0]                 # Phase 1 sweep
SDE_SOLVERS = ['euler', 'heun']

# One "family" per trained model of the linear path with the encoder start: (key, short, description, c, sigma, train name).
# 'S1' is the SI-schedules linear·enc model (gamma = 0, sigma = 0.10): Phase 0 extends its eps sweep.
FAMILIES = [('S1',   'S1 γ=0 σ=.10', 'γ = 0, σ = 0.10 (SI-schedules linear·enc)', 0.0, 0.10, 'si_sched_linear_enc_temp'),
            ('g0p5', 'γ=0.5',        'γ = 0.5·t(1−t), σ = 0.10',                  0.5, 0.10, 'si_noise_g0p5_enc_temp'),
            ('g1',   'γ=1',          'γ = 1·t(1−t), σ = 0.10',                    1.0, 0.10, 'si_noise_g1_enc_temp'),
            ('g2',   'γ=2',          'γ = 2·t(1−t), σ = 0.10',                    2.0, 0.10, 'si_noise_g2_enc_temp'),
            ('s0p25','σ=0.25',       'γ = 0, σ = 0.25',                           0.0, 0.25, 'si_noise_s0p25_enc_temp'),
            ('s0p5', 'σ=0.5',        'γ = 0, σ = 0.50',                           0.0, 0.50, 'si_noise_s0p5_enc_temp')]
FAM = {f[0]: dict(zip(['key', 'short', 'desc', 'c', 'sigma', 'train'], f)) for f in FAMILIES}
FAM_COLOR = {'S1': '#222222', 'g0p5': '#9ecae1', 'g1': '#4292c6', 'g2': '#08306b', 's0p25': '#fdae6b', 's0p5': '#d94801'}
SOLVER_MARKER = {'euler': 'o', 'heun': 's', 'rk4': '^', 'dopri5': 'D'}

def new_label(fam, solver, eps):
    return f'{FAM[fam]["short"]} | {solver} ε={eps:g}'

def grid(fam):
    if fam == 'S1':
        return ([(sv, e) for sv in SDE_SOLVERS for e in S1_EPS + P0_EPS] + [('rk4', 0.0), ('dopri5', 0.0)])
    return [(sv, e) for sv in SDE_SOLVERS for e in P1_EPS] + [('rk4', 0.0)]

# Previous runs, scored by the same compute_metrics on the same frames.
PREV = [('ref', 'Encoder alone',                         'encoder_only',                      'reference'),
        (0,     'DDPM (matched harness)',                'si_baseline_ddpm_matched_temp',     'DDPM'),
        (11,    'DiffusionSR (as published)',            'si_baseline_diffusionsr_temp',      'DDPM'),
        (12,    'prev SI-A s=0.10 enc, encoder start',   'si_si_opt_a_sigma010_encbase_temp', 'prev SI')]

arms = {}
for task, label, run, group in PREV:
    res = PREV_EVAL / run / f'{run}_results.json'
    if res.exists():
        skill = (PREV_EVAL / run / 'skill.json') if run == 'encoder_only' else (CACHE / run / 'skill.json')
        arms[label] = dict(task=task, run=run, group=group, results=res, skill=skill, eval_dir=PREV_EVAL / run,
                           artifact=f'{run}_eval_predictions')
    else:
        print(f'previous-run result missing, skipped: [{task}] {label}')

missing = []
for fam in FAM:
    train = FAM[fam]['train']
    for solver, eps in grid(fam):
        name = f'{train}__{solver}_{eps_tag(eps)}'
        for root, phase in [(SCHED_EVAL, 'S1'), (EVAL_ROOT, '0' if fam == 'S1' else '1')]:
            res = root / name / f'{name}_results.json'
            if res.exists():
                arms[new_label(fam, solver, eps)] = dict(
                    task=name, run=name, group=fam, results=res, skill=root / name / 'skill.json',
                    eval_dir=root / name, artifact=f'{name}_eval_predictions', fam=fam, phase=phase,
                    c=FAM[fam]['c'], sigma=FAM[fam]['sigma'], solver=solver, eps=eps, train=train)
                break
        else:
            missing.append(name)
NEW = [l for l in arms if 'fam' in arms[l]]
P0 = [l for l in NEW if arms[l]['phase'] == '0']
P1 = [l for l in NEW if arms[l]['phase'] == '1']
print(f'{len(arms) - len(NEW)} baselines; {len(NEW) - len(P0) - len(P1)} SI-schedules linear·enc evals, '
      f'{len(P0)} of 8 Phase-0 and {len(P1)} of 65 Phase-1 evals loaded'
      + (f'; {len(missing)} not evaluated yet (e.g. {missing[0]})' if missing else ''))

## 1 - Metrics

Notebook 21/22 columns, then the new ones: **CRPS** (K, lower is better), **SSR** (≈ 1 is best) over the melt pool and
all pixels, spread in the melt pool and NFE. Bold = best per column (calibration: nearest 0.95; SSR: nearest 1).

In [ ]:
# -- Prediction arrays: notebook 21's loader, plus the artifact name ----------------
def cached_arrays(run, eval_dir, artifact):
    d = CACHE / run
    if not ((d / 'pred.npy').exists() and (d / 'gt.npy').exists()):
        d.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(dir=CACHE, ignore_cleanup_errors=True) as tmp:
            src = eval_dir / 'predictions.npz'
            if not src.exists():
                import wandb
                wandb.Api().artifact(f'{WANDB_ENTITY}/{WANDB_PROJECT}/{artifact}:latest').download(root=tmp)
                src = Path(tmp) / 'predictions.npz'
            with np.load(src, allow_pickle=True) as z:
                np.save(d / 'pred.npy', z['pred'])
                np.save(d / 'gt.npy', z['gt'])
    return np.load(d / 'pred.npy', mmap_mode='r'), np.load(d / 'gt.npy', mmap_mode='r')

CACHE.mkdir(parents=True, exist_ok=True)
pred, gt = {}, None

def get_pred(label):
    """Load one arm's (N, K, C, H, W) predictions on first use; check every arm scores the same frames."""
    global gt
    if label not in pred:
        a = arms[label]
        p, g = cached_arrays(a['run'], a['eval_dir'], a['artifact'])
        if gt is None:
            gt = g
        elif g.shape != gt.shape or not np.array_equal(g[::500], gt[::500]):
            raise RuntimeError(f'{label}: ground truth differs from the other arms -- not the same test frames')
        pred[label] = p
    return pred[label]

def spread_of(p, g):
    """Notebook 21 section 6 / evaluate_si_schedules.ensemble_spread, streamed in chunks."""
    tot = tot_mp = 0.0; n_mp = 0
    for s in range(0, p.shape[0], 256):
        sd = np.asarray(p[s:s + 256, :, CH], dtype=np.float32).std(axis=1)
        mask = np.asarray(g[s:s + 256, CH]) > MP_THRESHOLD_K
        tot += float(sd.sum()); tot_mp += float(sd[mask].sum()); n_mp += int(mask.sum())
    return tot / sd[0].size / p.shape[0], tot_mp / max(n_mp, 1)

In [ ]:
# -- Metrics table -----------------------------------------------------------------
SHOW = ['MAE_mean', 'MAE_ensemble_mean', 'RMSE_mean', 'MP_MAE_mean', 'KH_MAE_mean', 'area_mae_um2',
        'calibration_2sigma', 'crps_melt_K', 'ssr_melt']
EXTRA = ['MAE_std', 'crps_all_K', 'ssr_all', 'spread_melt_px_K', 'spread_all_px_K', 'nfe_per_sample']
SKILL_KEYS = ['crps_all_K', 'crps_melt_K', 'ssr_all', 'ssr_melt', 'rank_hist_all', 'rank_hist_melt']
rows, rank_hist = {}, {}
for label, a in arms.items():
    r = json.load(open(a['results']))
    if 'crps_melt_K' not in r and Path(a['skill']).exists():
        r.update({k: v for k, v in json.load(open(a['skill'])).items() if k in SKILL_KEYS})
    rank_hist[label] = {k: r.get(f'rank_hist_{k}') for k in ('all', 'melt')}
    row = {'task': a['task'] if 'fam' not in a else '', 'group': a['group'], 'family': a.get('fam', ''),
           'phase': a.get('phase', ''), 'c': a.get('c', np.nan), 'sigma': a.get('sigma', np.nan),
           'solver': a.get('solver', ''), 'eps': a.get('eps', np.nan), **{k: r.get(k, np.nan) for k in SHOW + EXTRA}}
    if a['group'] == 'reference':
        row['calibration_2sigma'] = np.nan; row['ssr_melt'] = row['ssr_all'] = np.nan
    if a['group'] in ('DDPM', 'prev SI', 'reference'):
        row['nfe_per_sample'] = {'reference': 1, 'DDPM': 20, 'prev SI': 41}[a['group']]
        if a['group'] != 'reference':
            try:
                row['spread_all_px_K'], row['spread_melt_px_K'] = spread_of(get_pred(label), gt)
            except Exception as e:
                print(f'no predictions for {label} (spread left blank): {str(e).splitlines()[0][:100]}')
    rows[label] = row
metrics = pd.DataFrame(rows).T
for k in SHOW + EXTRA + ['eps', 'c', 'sigma']:
    metrics[k] = pd.to_numeric(metrics[k], errors='coerce')
no_skill = [l for l in arms if not np.isfinite(metrics.loc[l, 'crps_melt_K'])]
if no_skill:
    print(f'{len(no_skill)} arms have no skill scores yet (run slurm/si_noise/03_skill_existing.sh), e.g. {no_skill[0]}')
if EVAL_ROOT.exists():
    metrics.to_csv(EVAL_ROOT / 'all_results.csv')

TARGET = {'calibration_2sigma': 0.95, 'ssr_melt': 1.0, 'ssr_all': 1.0}
def _best(df, col):
    s = df[col].dropna()
    if s.empty:
        return None
    return (s - TARGET[col]).abs().idxmin() if col in TARGET else s.idxmin()

def show_table(df, cols=SHOW + ['spread_melt_px_K', 'nfe_per_sample']):
    best = {c: _best(df, c) for c in cols if c not in ('MAE_std', 'spread_melt_px_K', 'nfe_per_sample')}
    hl = lambda col: ['font-weight: bold; color: #0a7' if best.get(col.name) == i else '' for i in col.index]
    display(df[['group'] + cols].style.format({k: '{:.4g}' for k in cols}).apply(hl, axis=0, subset=cols))

BASELINES = [l for l in arms if 'fam' not in arms[l]]
print('Baselines and every model at deterministic Heun (eps = 0, 20 steps).')
show_table(metrics.loc[BASELINES + [new_label(f, 'heun', 0.0) for f in FAM if new_label(f, 'heun', 0.0) in arms]])

In [ ]:
# -- Did anything meet the success criterion? -----------------------------------------
REF_MP = metrics.loc['DiffusionSR (as published)', 'MP_MAE_mean'] if 'DiffusionSR (as published)' in metrics.index else 14.16
ok = metrics.loc[NEW]
ok = ok[(ok.calibration_2sigma >= 0.80) & (ok.ssr_melt.between(0.8, 1.25)) & (ok.MP_MAE_mean <= REF_MP)]
print(f'Success = calibration >= 0.80, 0.8 <= SSR (melt) <= 1.25, MP depth MAE <= {REF_MP:.2f} um (DiffusionSR).')
if ok.empty:
    print('No evaluation meets all three. Closest by CRPS in the melt pool:')
    show_table(metrics.loc[NEW].sort_values('crps_melt_K').head(10))
else:
    print(f'{len(ok)} evaluations meet all three, best CRPS (melt) first:')
    show_table(ok.sort_values('crps_melt_K'))

print('\nBest configuration per model and metric:')
HEAD = [('MAE_ensemble_mean', 'ensemble MAE (K)'), ('MP_MAE_mean', 'MP depth MAE (um)'), ('area_mae_um2', 'area MAE (um^2)'),
        ('calibration_2sigma', 'calibration (~0.95)'), ('crps_melt_K', 'CRPS melt (K)'), ('ssr_melt', 'SSR melt (~1)')]
best_rows = []
for f in FAM:
    sub = metrics[metrics.family == f]
    if sub.empty:
        continue
    row = {'model': FAM[f]['desc']}
    for m, name in HEAD:
        i = _best(sub, m)
        row[name] = f"{sub.loc[i, m]:.4g}  ({sub.loc[i, 'solver']}, ε={sub.loc[i, 'eps']:g})" if i else ''
    best_rows.append(row)
display(pd.DataFrame(best_rows))
print('\nAll evaluations (sortable copy saved as eval_results/si_noise/all_results.csv):')
show_table(metrics.loc[NEW].sort_values(['family', 'solver', 'eps']))

## 2 - Phase 0: much larger ε on the existing model

The SI-schedules `linear·enc` model (γ = 0, σ = 0.10), its notebook-22 sweep (ε ≤ 1) extended to ε = 20. On a Gaussian
with the exact field, larger ε also removes Euler's own 20-step under-dispersion (std 0.59 → 0.68 against 0.70; Heun is
already 0.69 at ε = 0), so a rising Euler curve here is partly the integrator, not only the model. Red lines: DDPM in
the matched harness (solid) and DiffusionSR as published (dotted); grey dotted: the ideal (0.95, SSR 1).

In [ ]:
# -- Metric vs eps: one figure helper for both phases ----------------------------------
SWEEP = [('MAE_ensemble_mean', 'ensemble MAE (K)'), ('MP_MAE_mean', 'MP depth MAE (um)'),
         ('calibration_2sigma', 'calibration'), ('crps_melt_K', 'CRPS, melt pool (K)'),
         ('ssr_melt', 'spread-skill, melt pool'), ('spread_melt_px_K', 'spread in melt pool (K)')]

def eps_sweep(fams, title):
    fig, axes = plt.subplots(2, 3, figsize=(15, 7.6), squeeze=False)
    for ax, (m, name) in zip(axes.flat, SWEEP):
        for f in fams:
            for sv in SDE_SOLVERS:
                sub = metrics[(metrics.family == f) & (metrics.solver == sv)].sort_values('eps')
                if len(sub):
                    ax.plot(sub.eps, sub[m], '-' if sv == 'heun' else '--', marker=SOLVER_MARKER[sv], ms=4,
                            color=FAM_COLOR[f], label=f'{FAM[f]["short"]}, {sv}')
        for lab, sty in [('DDPM (matched harness)', '-'), ('DiffusionSR (as published)', ':')]:
            if lab in metrics.index and np.isfinite(metrics.loc[lab, m]):
                ax.axhline(metrics.loc[lab, m], color='#cc3333', ls=sty, lw=1)
        if m in TARGET:
            ax.axhline(TARGET[m], color='grey', ls=':', lw=1)
        ax.set_xscale('symlog', linthresh=0.01); ax.grid(alpha=0.3)
        ax.set_title(name, fontsize=10); ax.set_xlabel('ε')
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc='lower center', ncol=min(len(l), 6), fontsize=8, bbox_to_anchor=(0.5, -0.03))
    fig.suptitle(title, fontsize=11)
    plt.tight_layout(rect=(0, 0.04, 1, 1)); plt.show()

eps_sweep(['S1'], 'Phase 0: SI-schedules linear·enc, ε = 0 ... 20 (solid = Heun, dashed = Euler)')

## 3 - Phase 1: latent noise γ and larger base noise σ

Left family: γ_t = c·t(1−t) at σ = 0.10 (blues, darker = larger c), with the SI-schedules γ = 0 model (black) as c = 0.
Right family: γ = 0 with σ = 0.10 (black), 0.25 and 0.5 (oranges). Each model is ONE training run; ε is sampling-only.

In [ ]:
# -- Phase 1 eps sweeps -----------------------------------------------------------------
eps_sweep(['S1', 'g0p5', 'g1', 'g2'], 'Latent noise γ = c·t(1−t), σ = 0.10 (black = c = 0; solid = Heun, dashed = Euler)')
eps_sweep(['S1', 's0p25', 's0p5'], 'Base noise σ, γ = 0 (black = σ = 0.10; solid = Heun, dashed = Euler)')

## 4 - Accuracy against honest spread

Each line follows one (model, solver) as ε rises. Top: CRPS in the melt pool against melt-pool depth MAE (both lower is
better: the bottom-left corner wins). Bottom: SSR against ensemble MAE; the target is SSR ≈ 1 at low error. Stars and
crosses mark DDPM, DiffusionSR and the previous best interpolant.

In [ ]:
# -- Frontier -------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
for ax, (xm, ym, xname, yname) in zip(axes, [('MP_MAE_mean', 'crps_melt_K', 'MP depth MAE (um)', 'CRPS, melt pool (K)'),
                                             ('ssr_melt', 'MAE_ensemble_mean', 'spread-skill ratio, melt pool', 'ensemble MAE (K)'),
                                             ('calibration_2sigma', 'MP_MAE_mean', 'calibration', 'MP depth MAE (um)')]):
    for f in FAM:
        for sv in SDE_SOLVERS:
            sub = metrics[(metrics.family == f) & (metrics.solver == sv)].sort_values('eps')
            if len(sub):
                ax.plot(sub[xm], sub[ym], '-' if sv == 'heun' else '--', marker=SOLVER_MARKER[sv], ms=4,
                        color=FAM_COLOR[f], alpha=0.9, label=f'{FAM[f]["short"]}, {sv}')
                ax.annotate(f"ε={sub.eps.iloc[-1]:g}", (sub[xm].iloc[-1], sub[ym].iloc[-1]), fontsize=6)
    for lab, mk, col in [('DDPM (matched harness)', '*', '#cc3333'), ('DiffusionSR (as published)', 'P', '#cc3333'),
                         ('prev SI-A s=0.10 enc, encoder start', 'X', '#888888'), ('Encoder alone', 'o', '#555555')]:
        if lab in metrics.index and np.isfinite(metrics.loc[lab, xm]) and np.isfinite(metrics.loc[lab, ym]):
            ax.scatter(metrics.loc[lab, xm], metrics.loc[lab, ym], color=col, marker=mk, s=120, zorder=5, label=lab)
    if xm in TARGET:
        ax.axvline(TARGET[xm], color='grey', ls=':', lw=1)
    ax.set_xlabel(xname); ax.set_ylabel(yname); ax.grid(alpha=0.3)
axes[0].legend(fontsize=6, ncol=2)
plt.tight_layout(); plt.show()

## 5 - Rank histograms

Where the truth falls among each ensemble's 10 sorted samples, inside the true melt pool, normalised so a perfectly
consistent ensemble gives the flat dotted line at 1/11. A U shape means the truth too often lies outside the ensemble
(under-dispersed); a central hump means the ensemble is wider than its errors (over-dispersed).

In [ ]:
# -- Rank histograms ------------------------------------------------------------------------
def best_of(fam, col='crps_melt_K'):
    sub = metrics[metrics.family == fam][col].dropna()
    return sub.idxmin() if len(sub) else None

RH_ARMS = ['DDPM (matched harness)', 'DiffusionSR (as published)', 'prev SI-A s=0.10 enc, encoder start',
           new_label('S1', 'heun', 0.0), new_label('S1', 'euler', 20.0)] + [best_of(f) for f in FAM if f != 'S1']
RH_ARMS = [l for l in dict.fromkeys(RH_ARMS) if l and rank_hist.get(l, {}).get('melt')]
if RH_ARMS:
    ncol = 4
    nrow = -(-len(RH_ARMS) // ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 2.9 * nrow), squeeze=False)
    for ax, l in zip(axes.flat, RH_ARMS):
        h = np.asarray(rank_hist[l]['melt'], dtype=float)
        h /= h.sum()
        ax.bar(np.arange(len(h)), h, color='#4292c6' if 'fam' in arms[l] else '#cc3333')
        ax.axhline(1 / len(h), color='grey', ls=':', lw=1)
        ax.set_title(f"{l}\nSSR {metrics.loc[l, 'ssr_melt']:.2f}, CRPS {metrics.loc[l, 'crps_melt_K']:.0f} K", fontsize=8)
        ax.set_xticks([0, len(h) - 1]); ax.set_xticklabels(['below all', 'above all'], fontsize=7)
    for ax in list(axes.flat)[len(RH_ARMS):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('no rank histograms yet')

## 6 - Training

One run per Phase-1 model, with the SI-schedules γ = 0, σ = 0.10 run in black. Loss values are comparable within a
family only loosely: γ and σ change the size of the denoiser target's noise and of the velocity target. Dots mark the
best (lowest validation) epoch, the checkpoint every eval used.

In [ ]:
# -- Loss curves ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
summary = {}
for f in FAM:
    d = (SCHED_RUNS if f == 'S1' else RUNS_ROOT) / FAM[f]['train']
    for ax, fname in zip(axes, ['loss_epoch.txt', 'validation_loss_epoch.txt']):
        p = d / fname
        if not p.exists():
            continue
        y = np.loadtxt(p).reshape(-1)
        ax.plot(y, lw=1, color=FAM_COLOR[f], label=FAM[f]['desc'])
        if 'validation' in fname:
            k = int(np.argmin(y)); ax.plot(k, y[k], 'o', ms=4, color=FAM_COLOR[f])
            summary[FAM[f]['desc']] = {'epochs': len(y), 'best epoch': k, 'best val loss': y[k], 'final / best': y[-1] / y[k]}
for ax, kind in zip(axes, ['train', 'validation']):
    ax.set_yscale('log'); ax.set_title(kind); ax.set_xlabel('epoch'); ax.grid(alpha=0.3)
axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
if summary:
    display(pd.DataFrame(summary).T.style.format({'best val loss': '{:.4g}', 'final / best': '{:.2f}'}))

## 7 - Overlays

Notebook 21/22's overlay, unchanged (shared frame, fixed 293–5000 K scale, p99-clipped error and std scales shared by
every row). Rows: the baselines, then each model's best evaluation by CRPS in the melt pool. Edit `SHOW_ARMS` to change
them; arrays are fetched on first use.

In [ ]:
# -- Overlay arms and frame selection --------------------------------------------------
SHOW_ARMS = (['Encoder alone', 'DDPM (matched harness)', 'DiffusionSR (as published)', 'prev SI-A s=0.10 enc, encoder start']
             + [best_of(f) for f in FAM])
SHOW_ARMS = [l for l in dict.fromkeys(SHOW_ARMS) if l in arms]
for l in SHOW_ARMS:
    try:
        get_pred(l)
    except Exception as e:
        print(f'no predictions for {l}: {str(e).splitlines()[0][:120]}')

N_FRAMES = 0 if gt is None else gt.shape[0]
HOT = (np.asarray(gt[:, CH]) > MP_THRESHOLD_K).sum(axis=(1, 2)) if N_FRAMES else np.array([])
DEFAULT_FRAME = int(np.argmax(HOT)) if N_FRAMES else 0
print(f'{N_FRAMES} test frames, {int((HOT > 0).sum())} with a melt pool; default frame {DEFAULT_FRAME}')

def with_frame_slider(render_fn, default=DEFAULT_FRAME):
    if N_FRAMES == 0:
        return
    if not HAVE_WIDGETS:
        render_fn(default)
        return
    slider = widgets.IntSlider(min=0, max=N_FRAMES - 1, value=default, description='frame:',
                               continuous_update=False, layout=widgets.Layout(width='460px'))
    display(widgets.VBox([slider, widgets.interactive_output(render_fn, {'frame': slider})]))

def row_label(l):
    a = arms[l]
    return f"[{a['task']}] {l}" if 'fam' not in a else l

In [ ]:
# -- Rebuilt overlay: shared frame, shared scales (notebook 21 section 4) -------------
N_SAMPLES_SHOWN = 3

def render_overlay(frame, show=None):
    labels = [l for l in (show or SHOW_ARMS) if l in pred]
    if not labels:
        return
    g = np.asarray(gt[frame, CH])
    P = {l: np.asarray(pred[l][frame, :, CH]) for l in labels}
    err_max = float(np.percentile(np.stack([np.abs(P[l].mean(0) - g) for l in labels]), 99)) or 1.0
    std_max = float(np.percentile(np.stack([P[l].std(0) for l in labels]), 99)) or 1.0
    ncol = 1 + N_SAMPLES_SHOWN + 3
    fig, axes = plt.subplots(len(labels), ncol, figsize=(2.2 * ncol, 2.2 * len(labels)), squeeze=False)
    col_titles = (['ground truth'] + [f'sample {k + 1}' for k in range(N_SAMPLES_SHOWN)]
                  + ['ensemble mean', f'|mean - GT|, 0-{err_max:.0f} K (p99)', f'ensemble std, 0-{std_max:.0f} K (p99)'])
    for r, l in enumerate(labels):
        p, m = P[l], P[l].mean(0)
        panels = [(g, 'jet', VMIN_T, VMAX_T)]
        panels += [(p[k], 'jet', VMIN_T, VMAX_T) if k < len(p) else None for k in range(N_SAMPLES_SHOWN)]
        panels += [(m, 'jet', VMIN_T, VMAX_T), (np.abs(m - g), 'magma', 0, err_max), (p.std(0), 'hot', 0, std_max)]
        for c, panel in enumerate(panels):
            ax = axes[r, c]
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(col_titles[c], fontsize=8)
            if panel is None:
                ax.axis('off'); continue
            img, cmap, vmin, vmax = panel
            ax.imshow(img.T, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax)
        axes[r, 0].set_ylabel(row_label(l), fontsize=7)
    fig.suptitle(f'Frame {frame} - temperature (K), fixed scale {VMIN_T}-{VMAX_T} K', fontsize=10)
    plt.tight_layout(); plt.show()

with_frame_slider(render_overlay)

In [ ]:
# -- What eps does to the samples: one model, Heun, every eps (same overlay settings) ---
EPS_FAM = 'g1'          # edit: 'S1', 'g0p5', 'g1', 'g2', 's0p25', 's0p5'
EPS_ROWS = [l for l in metrics[(metrics.family == EPS_FAM) & (metrics.solver == 'heun')].sort_values('eps').index]
for l in EPS_ROWS:
    try:
        get_pred(l)
    except Exception as e:
        print(f'no predictions for {l}: {str(e).splitlines()[0][:120]}')
with_frame_slider(lambda frame: render_overlay(frame, show=['DDPM (matched harness)'] + EPS_ROWS))

## 8 - Melt-pool boundaries

Notebook 21 section 5, unchanged: the 1900 K isotherm of each arm's ensemble mean against the ground truth (black), and
every individual sample's boundary for the arms in `SPREAD_ARMS`.

In [ ]:
# -- Melt-pool contours ----------------------------------------------------------------
PALETTE = ['#00d5ff', '#cc3333', '#ff7f0e', '#3366cc', '#9467bd', '#00aa77', '#cc8800', '#17becf']
SPREAD_ARMS = ['DDPM (matched harness)', new_label('S1', 'heun', 0.0), best_of('g1'), best_of('s0p5')]
SPREAD_ARMS = [l for l in dict.fromkeys(SPREAD_ARMS) if l in arms]
for l in SPREAD_ARMS:
    try:
        get_pred(l)
    except Exception as e:
        print(f'no predictions for {l}: {str(e).splitlines()[0][:120]}')

def _crop(ax, g, margin=6):
    hot = np.argwhere(g > MP_THRESHOLD_K)
    if len(hot):
        (x0, z0), (x1, z1) = hot.min(0), hot.max(0)
        ax.set_xlim(max(x0 - margin, 0) - 0.5, min(x1 + margin, g.shape[0] - 1) + 0.5)
        ax.set_ylim(max(z0 - margin, 0) - 0.5, min(z1 + margin, g.shape[1] - 1) + 0.5)

def render_contours(frame):
    g = np.asarray(gt[frame, CH])
    spread = [l for l in SPREAD_ARMS if l in pred]
    fig, axes = plt.subplots(1, 1 + len(spread), figsize=(6 * (1 + len(spread)), 5.5), squeeze=False)
    axes = axes[0]
    for ax in axes:
        ax.imshow(g.T, origin='lower', cmap='Greys_r', vmin=VMIN_T, vmax=VMAX_T)
        if g.max() > MP_THRESHOLD_K:
            ax.contour(g.T, levels=[MP_THRESHOLD_K], colors='k', linewidths=2.5)
        _crop(ax, g)
    handles = [plt.Line2D([], [], color='k', lw=2.5, label='ground truth')]
    for i, l in enumerate([x for x in SHOW_ARMS if x in pred]):
        m = np.asarray(pred[l][frame, :, CH]).mean(0)
        col = PALETTE[i % len(PALETTE)]
        if m.max() > MP_THRESHOLD_K:
            axes[0].contour(m.T, levels=[MP_THRESHOLD_K], colors=[col], linewidths=1.3)
        handles.append(plt.Line2D([], [], color=col, label=row_label(l)))
    axes[0].legend(handles=handles, fontsize=7, loc='lower left')
    axes[0].set_title(f'Frame {frame}: 1900 K boundary of each ensemble mean', fontsize=10)
    for ax, l in zip(axes[1:], spread):
        p = np.asarray(pred[l][frame, :, CH])
        for k in range(len(p)):
            if p[k].max() > MP_THRESHOLD_K:
                ax.contour(p[k].T, levels=[MP_THRESHOLD_K], colors=['#cc3333'], linewidths=0.8, alpha=0.7)
        ax.set_title(f'{row_label(l)}: all {len(p)} samples (red)', fontsize=9)
    plt.tight_layout(); plt.show()

with_frame_slider(render_contours)

## 9 - Ensemble spread

Per-pixel ensemble std maps on one shared scale (clipped at p99), then the test-set averages next to calibration, SSR
and CRPS.

In [ ]:
# -- Per-pixel ensemble std, same slider ----------------------------------------------
ENS_ARMS = [l for l in SHOW_ARMS if l in pred and pred[l].shape[1] > 1]

def render_std(frame):
    S = {l: np.asarray(pred[l][frame, :, CH]).std(0) for l in ENS_ARMS}
    if not S:
        return
    vmax = float(np.percentile(np.stack(list(S.values())), 99)) or 1.0
    fig, axes = plt.subplots(1, len(S), figsize=(3.1 * len(S), 3.3), squeeze=False)
    for ax, (l, s) in zip(axes[0], S.items()):
        im = ax.imshow(s.T, origin='lower', cmap='hot', vmin=0, vmax=vmax)
        ax.set_title(row_label(l), fontsize=7); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=axes[0].tolist(), fraction=0.02, label='std (K)')
    fig.suptitle(f'Frame {frame}: per-pixel ensemble std, shared scale 0-{vmax:.0f} K (clipped at p99)', fontsize=10)
    plt.show()

with_frame_slider(render_std)

cols = ['spread_all_px_K', 'spread_melt_px_K', 'calibration_2sigma', 'ssr_melt', 'crps_melt_K', 'MAE_ensemble_mean']
tab = metrics.loc[[l for l in SHOW_ARMS if arms[l]['group'] != 'reference'], cols]
tab.columns = ['mean std, all px (K)', 'mean std, melt px (K)', 'calibration_2sigma', 'SSR melt', 'CRPS melt (K)', 'ensemble MAE (K)']
display(tab.style.format('{:.3g}'))

## 10 - Head-to-head

DDPM in the matched harness (task 0), DiffusionSR as published (task 11), the SI-schedules best by ensemble MAE
(linear·enc) and this study's best by **CRPS in the melt pool**: every sample, ensemble mean, error and spread, cropped to
the melt region, for the largest, a median and a small melt pool.

In [ ]:
# -- Head-to-head metrics ----------------------------------------------------------------
S1_BEST = metrics.loc[[l for l in NEW if arms[l]['fam'] == 'S1'], 'MAE_ensemble_mean'].idxmin() if NEW else None
BEST_NEW = metrics.loc[P0 + P1, 'crps_melt_K'].dropna().idxmin() if metrics.loc[P0 + P1, 'crps_melt_K'].notna().any() else None
H2H = [l for l in dict.fromkeys(['DDPM (matched harness)', 'DiffusionSR (as published)', S1_BEST, BEST_NEW]) if l in arms]
H2H_COLOR = dict(zip(H2H, ['#cc3333', '#ff7f0e', '#888888', '#3366cc']))
for l in H2H:
    get_pred(l)
H2H_METRICS = [('MAE_ensemble_mean', 'ensemble MAE (K)'), ('MAE_mean', 'per-sample MAE (K)'), ('RMSE_mean', 'RMSE (K)'),
               ('MP_MAE_mean', 'melt-pool depth MAE (um)'), ('KH_MAE_mean', 'keyhole depth MAE (um)'),
               ('area_mae_um2', 'melt-pool area MAE (um^2)'), ('calibration_2sigma', 'calibration (ideal ~0.95)'),
               ('spread_melt_px_K', 'sample spread in melt pool (K)'), ('crps_melt_K', 'CRPS, melt pool (K)'),
               ('ssr_melt', 'spread-skill ratio, melt pool (~1)'), ('nfe_per_sample', 'network evals / sample')]
base = H2H[0]
table = {}
for l in H2H:
    col = {}
    for m, name in H2H_METRICS:
        v, v0 = metrics.loc[l, m], metrics.loc[base, m]
        col[name] = f'{v:.4g}' if l == base else f'{v:.4g}  ({100.0 * (v - v0) / v0:+.0f}%)'
    table[row_label(l)] = col
print('Brackets: change relative to task 0. Lower is better for every error row; calibration is best near 0.95.')
display(pd.DataFrame(table))

melted = np.where(HOT > 0)[0]
by_size = melted[np.argsort(HOT[melted], kind='stable')]
H2H_FRAMES = {'largest melt pool': int(by_size[-1]), 'median melt pool': int(by_size[len(by_size) // 2]),
              'small melt pool': int(by_size[len(by_size) // 10])}

def show_frames(render_fn):
    if HAVE_WIDGETS:
        with_frame_slider(render_fn)
    else:
        for f in H2H_FRAMES.values():
            render_fn(f)

def _melt_box(g, margin=6):
    hot = np.argwhere(g > MP_THRESHOLD_K)
    if not len(hot):
        return None
    (x0, z0), (x1, z1) = hot.min(0), hot.max(0)
    return (max(x0 - margin, 0) - 0.5, min(x1 + margin, g.shape[0] - 1) + 0.5,
            max(z0 - margin, 0) - 0.5, min(z1 + margin, g.shape[1] - 1) + 0.5)

def render_h2h_overlay(frame):
    g = np.asarray(gt[frame, CH])
    P = {l: np.asarray(pred[l][frame, :, CH]) for l in H2H}
    K = max(p.shape[0] for p in P.values())
    err_max = float(np.percentile(np.stack([np.abs(P[l].mean(0) - g) for l in H2H]), 99)) or 1.0
    std_max = float(np.percentile(np.stack([P[l].std(0) for l in H2H]), 99)) or 1.0
    box = _melt_box(g)
    titles = (['ground truth'] + [f'sample {k + 1}' for k in range(K)]
              + ['ensemble mean', f'|mean - GT|\n0-{err_max:.0f} K (p99)', f'ensemble std\n0-{std_max:.0f} K (p99)'])
    fig, axes = plt.subplots(len(H2H), len(titles), figsize=(1.55 * len(titles), 1.9 * len(H2H) + 0.6), squeeze=False)
    for r, l in enumerate(H2H):
        p, m = P[l], P[l].mean(0)
        imgs = ([(g, 'jet', VMIN_T, VMAX_T)] + [(p[k], 'jet', VMIN_T, VMAX_T) for k in range(len(p))]
                + [None] * (K - len(p))
                + [(m, 'jet', VMIN_T, VMAX_T), (np.abs(m - g), 'magma', 0, err_max), (p.std(0), 'hot', 0, std_max)])
        for c, panel in enumerate(imgs):
            ax = axes[r, c]
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(titles[c], fontsize=7)
            if panel is None:
                ax.axis('off'); continue
            img, cmap, vmin, vmax = panel
            ax.imshow(img.T, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax)
            if box:
                ax.set_xlim(box[0], box[1]); ax.set_ylim(box[2], box[3])
        axes[r, 0].set_ylabel(row_label(l), fontsize=6, color=H2H_COLOR[l])
    name = next((k for k, f in H2H_FRAMES.items() if f == frame), 'selected frame')
    fig.suptitle(f'Frame {frame} ({name}): every sample, fixed scale {VMIN_T}-{VMAX_T} K', fontsize=10)
    plt.tight_layout(); plt.show()

show_frames(render_h2h_overlay)

In [ ]:
# -- Head-to-head melt-pool boundaries -------------------------------------------------
def render_h2h_contours(frame):
    g = np.asarray(gt[frame, CH])
    box = _melt_box(g)
    fig, axes = plt.subplots(1, 1 + len(H2H), figsize=(4.6 * (1 + len(H2H)), 4.6))
    for ax in axes:
        ax.imshow(g.T, origin='lower', cmap='Greys_r', vmin=VMIN_T, vmax=VMAX_T)
        if g.max() > MP_THRESHOLD_K:
            ax.contour(g.T, levels=[MP_THRESHOLD_K], colors='k', linewidths=2.5)
        if box:
            ax.set_xlim(box[0], box[1]); ax.set_ylim(box[2], box[3])
    handles = [plt.Line2D([], [], color='k', lw=2.5, label='ground truth')]
    for l in H2H:
        m = np.asarray(pred[l][frame, :, CH]).mean(0)
        if m.max() > MP_THRESHOLD_K:
            axes[0].contour(m.T, levels=[MP_THRESHOLD_K], colors=[H2H_COLOR[l]], linewidths=1.6)
        handles.append(plt.Line2D([], [], color=H2H_COLOR[l], lw=1.6, label=row_label(l)))
    axes[0].legend(handles=handles, fontsize=6, loc='lower left')
    axes[0].set_title('ensemble-mean boundary (1900 K)', fontsize=10)
    for ax, l in zip(axes[1:], H2H):
        p = np.asarray(pred[l][frame, :, CH])
        for k in range(len(p)):
            if p[k].max() > MP_THRESHOLD_K:
                ax.contour(p[k].T, levels=[MP_THRESHOLD_K], colors=[H2H_COLOR[l]], linewidths=0.8, alpha=0.75)
        ax.set_title(f'all {len(p)} sample boundaries', fontsize=9, color=H2H_COLOR[l])
    name = next((k for k, f in H2H_FRAMES.items() if f == frame), 'selected frame')
    fig.suptitle(f'Frame {frame} ({name}): black = ground truth', fontsize=10)
    plt.tight_layout(); plt.show()

show_frames(render_h2h_contours)

## 11 - Reading this correctly

- **What is held equal.** Test frames, K = 10, 20 steps, metric code, the frozen encoder and its per-step conditioning,
  data, harness and every training hyperparameter. Within Phase 1 the trained variables are γ's scale c and the base
  noise σ; solver and ε are sampling-only and share one checkpoint per model.
- **ε means the same as in notebook 22** (ε_t = ε·Γ_t²), but with γ > 0 or σ > 0.1, Γ_t is larger, so the same ε now
  injects real noise. Compare models at equal ε only with that in mind; compare them at their own best ε for the verdict.
- **With an exact model, ε does not change the final distribution**, only the path to it. What ε (and γ, σ) can change
  is how forgiving sampling is of the network's errors. More spread is only good when CRPS falls and SSR moves toward 1
  together; spread that raises CRPS is error, not uncertainty.
- **Euler at 20 steps under-disperses on its own** (exact-field Gaussian: std 0.59 vs 0.70 for γ = 0, 0.49 for c = 2 at
  ε = 0; Heun 0.69 / 0.60). Larger ε corrects this, so for Euler part of the ε effect is the integrator.
- **CRPS, SSR and rank histograms are per pixel.** They do not judge whether a sample's spatial structure is physical;
  the overlays and boundary plots do.
- **Back-filled scores** for the baselines and the SI-schedules evals come from their saved predictions with the same
  function the new evals use (`evaluate_si_schedules.ensemble_skill`).
- **Physics metrics in µm assume 10 µm per HR pixel**, as in notebooks 21 and 22; rankings do not depend on it.